# ICAIF 2024 금융-RAG 챌린지 기본 예제

이 노트북은 **ICAIF 2024 금융-RAG 챌린지**를 위한 **기본 예제**입니다. 이 챌린지의 목표는 금융 데이터를 위한 **Retrieval-Augmented Generation (RAG)** 시스템을 만드는 것입니다. 참가자는 대규모 코퍼스에서 관련 문서를 검색하고 사용자 Query에 대한 정확하고 상황에 맞는 응답을 제공하는 시스템을 개발해야 합니다.

---

## 시스템 구성 요소

기본 예제의 시스템은 두 가지 주요 구성 요소로 나뉩니다:

1. **검색**: 사용자 쿼리를 기반으로 대규모 금융 문서 코퍼스에서 관련 문서를 검색합니다.
2. **재정렬**: 검색된 문서의 순위를 다시 매겨 가장 관련성 높은 정보가 우선되도록 합니다.

---

## 모델 개요

이 베이스라인 노트북은 `SentenceTransformer`와 `CrossEncoder` 모델을 조합하여 다음 작업을 수행합니다:

- **검색 모델**은 쿼리와 문서를 임베딩으로 인코딩하는 역할을 담당합니다.
- **재정렬 모델**은 검색된 문서의 관련성을 평가하고 순서를 조정합니다.

이 예시에서는 **FinDER**라는 FinanceRAG 프로젝트의 7개 과제 중 하나를 사용합니다. 검색 모델로는 `intfloat/e5-large-v2`가 사용되며, 재정렬은 `cross-encoder/ms-marco-MiniLM-L-12-v2`를 통해 수행됩니다. 두 모델 모두 `sentence_transformers` 라이브러리에서 지원하는 다른 모델로 대체하여 성능을 실험해볼 수 있습니다.

---

## 목표

이 노트북의 목표는 참가자들이 챌린지를 위한 보다 **고급 솔루션**을 구축할 수 있는 **탄탄한 기반**을 제공하는 것입니다. 과제, 검색 모델 및 재정렬 모델을 필요에 따라 자유롭게 개발하세요!

---

## Repository Setup and Environment Configuration

GitHub 리포지토리 확인 [here](https://github.com/linq-rag/FinanceRAG).

아래와 같이 Github repository를 Clone하기:

### 1. Clone the repository:

```bash
git clone https://github.com/linq-rag/FinanceRAG.git
cd FinanceRAG
```

### 2. Set up the Python environment:

#### If using `venv` (Python 3.11 or higher required):

```bash
python3 -m venv .venv
source .venv/bin/activate  # On Windows use .venv\Scriptsctivate
pip install --upgrade pip
pip install -r requirements.txt
```

#### If using `conda`:

```bash
conda create -n financerag python=3.11
conda activate financerag
pip install -r requirements.txt
```

준비가 완료되었습니다!

In [1]:
# Step 1: Import necessary libraries
# --------------------------------------
# Import required libraries for document retrieval, reranking, and logging setup.
from sentence_transformers import CrossEncoder
import logging

from financerag.rerank import CrossEncoderReranker
from financerag.retrieval import DenseRetrieval, SentenceTransformerEncoder
from financerag.tasks import FinDER

# Setup basic logging configuration to show info level messages.
logging.basicConfig(level=logging.INFO)


c:\Users\Jeawon\Desktop\2024데이터경진\data-contest-2024F\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
# Step 2: Initialize FinDER Task
# --------------------------
# In this baseline example, we are using the FinDER task, one of the seven available tasks in this project.
# If you want to use a different task, for example, 'OtherTask', you can change the task initialization as follows:
#
# Example:
# from financerag.tasks import OtherTask
# finder_task = OtherTask()
#
# For this baseline, we proceed with FinDER.
finder_task = FinDER()


INFO:financerag.common.loader:Loading Corpus...
INFO:financerag.common.loader:Loaded 13867 Documents.
INFO:financerag.common.loader:Corpus Example: {'id': 'ADBE20230004', 'title': 'ADBE OVERVIEW', 'text': 'Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsi

In [5]:
print("Queries:", list(finder_task.queries.items())[:5])  # 처음 5개의 쿼리 출력
print("Corpus:", list(finder_task.corpus.items())[:5])    

Queries: [('q00001', 'What are the service and product offerings from Microsoft'), ('q00002', 'MSFT segment breakdown'), ('q00003', 'Who are Microsoft`s key customers?'), ('q00004', 'What is Microsoft`s business model'), ('q00005', 'MSFT Capex commitment')]
Corpus: [('ADBE20230004', {'title': 'ADBE OVERVIEW', 'text': 'Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators b

In [3]:
len(list(finder_task.queries.items()))
len(list(finder_task.corpus.items()))

13863

In [3]:
finder_task

In [ ]:
# Step 3: Initialize DenseRetriever model
# -------------------------------------
# Initialize the retrieval model using SentenceTransformers. This model will be responsible
# for encoding both the queries and documents into embeddings.
#
# You can replace 'intfloat/e5-large-v2' with any other model supported by SentenceTransformers.
# For example: 'BAAI/bge-large-en-v1.5', 'Linq-AI-Research/Linq-Embed-Mistral', etc.
encoder_model = SentenceTransformerEncoder(
    model_name_or_path='intfloat/e5-large-v2',
    query_prompt='query: ',
    doc_prompt='passage: ',
)

retrieval_model = DenseRetrieval(
    model=encoder_model
)


In [ ]:
# Step 4: Perform retrieval
# ---------------------
# Use the model to retrieve relevant documents for given queries.
retrieval_model = DenseRetrieval(
    model=encoder_model
)

retrieval_result = finder_task.retrieve(
    retriever=retrieval_model
)

# Print a portion of the retrieval results to verify the output.
print(f"Retrieved results for {len(retrieval_result)} queries. Here's an example of the top 5 documents for the first query:")

for q_id, result in retrieval_result.items():
    print(f"\nQuery ID: {q_id}")
    # Sort the result to print the top 5 document ID and its score
    sorted_results = sorted(result.items(), key=lambda x: x[1], reverse=True)

    for i, (doc_id, score) in enumerate(sorted_results[:5]):
        print(f"  Document {i + 1}: Document ID = {doc_id}, Score = {score}")

    break  # Only show the first query


In [ ]:
# Step 5: Initialize CrossEncoder Reranker
# --------------------------------------
# The CrossEncoder model will be used to rerank the retrieved documents based on relevance.
#
# You can replace 'cross-encoder/ms-marco-MiniLM-L-12-v2' with any other model supported by CrossEncoder.
# For example: 'cross-encoder/ms-marco-TinyBERT-L-2', 'cross-encoder/stsb-roberta-large', etc.
reranker = CrossEncoderReranker(
    model=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
)


In [ ]:
# Step 6: Perform reranking
# -------------------------
# Rerank the top 100 retrieved documents using the CrossEncoder model.
reranking_result = finder_task.rerank(
    reranker=reranker,
    results=retrieval_result,
    top_k=100,  # Rerank the top 100 documents
    batch_size=32
)

# Print a portion of the reranking results to verify the output.
print(f"Reranking results for {len(reranking_result)} queries. Here's an example of the top 5 documents for the first query:")

for q_id, result in reranking_result.items():
    print(f"\nQuery ID: {q_id}")
    # Sort the result to print the top 5 document ID and its score
    sorted_results = sorted(result.items(), key=lambda x: x[1], reverse=True)

    for i, (doc_id, score) in enumerate(sorted_results[:5]):
        print(f"  Document {i + 1}: Document ID = {doc_id}, Score = {score}")

    break  # Only show the first query


In [ ]:
# Step 7: Save results
# -------------------
# Save the results to the specified output directory as a CSV file.
output_dir = './results'
finder_task.save_results(output_dir=output_dir)

# Confirm the results have been saved.
print(f"Results have been saved to {output_dir}/FinDER/results.csv")


# OPEN AI Embadding

In [10]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [11]:
import os
import openai

# OpenAI API 키 설정
openai.api_key = os.getenv("OPENAI_API_KEY")

In [17]:
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("2024DATA-science-Retriever")

LangSmith 추적을 시작합니다.
[프로젝트명]
2024DATA-science-Retriever


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [ ]:
from langchain_openai import OpenAIEmbeddings

# OpenAI의 "text-embedding-3-large" 모델을 사용하여 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# OpenAI 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings()


In [14]:
# import csv
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity
# from langchain_openai import OpenAIEmbeddings

# # 임베딩 벡터 생성 함수
# def get_embedding(text, model="text-embedding-3-small"):
#     embeddings = OpenAIEmbeddings(model=model)
#     response = embeddings.embed_query(text)
#     return response

# # # 샘플 데이터 (쿼리 및 코퍼스)
# # queries = [
# #     ("q00001", "What are the service and product offerings from Microsoft"),
# #     ("q00002", "MSFT segment breakdown"),
# #     # 필요한 만큼 쿼리를 추가하세요
# # ]

# # corpus = [
# #     (
# #         "ADBE20230004",
# #         {
# #             "title": "ADBE OVERVIEW",
# #             "text": "Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsible. Our products and services help unleash creativity, accelerate document productivity and power businesses in a digital world.",
# #         },
# #     ),
# #     (
# #         "ADBE20230006",
# #         {
# #             "title": "ADBE OFFERINGS",
# #             "text": "We deliver a wide range of products, services and solutions to empower our customers and users to imagine and express ideas, create content and bring any digital experience to life. We focus our strategic investments in two areas of growth:",
# #         },
# #     ),
# #     # 필요한 만큼 코퍼스 문서를 추가하세요
# # ]
# queries = list(finder_task.queries.items())
# corpus = list(finder_task.corpus.items())


# # 쿼리 및 코퍼스에 대해 임베딩 벡터 생성 및 CSV 저장
# with open("query_embeddings.csv", mode="w", newline="") as file:
#     writer = csv.writer(file)
#     writer.writerow(["query_id", "embedding"])
#     for qid, text in queries:
#         embedding = get_embedding(text)
#         writer.writerow([qid, embedding])

# with open("corpus_embeddings.csv", mode="w", newline="") as file:
#     writer = csv.writer(file)
#     writer.writerow(["corpus_id", "embedding"])
#     for cid, content in corpus:
#         combined_text = f"{content['title']} {content['text']}"  # title과 text 결합
#         embedding = get_embedding(combined_text)
#         writer.writerow([cid, embedding])

# # 임베딩 벡터 불러오기
# def load_embeddings(file_path):
#     embeddings = {}
#     with open(file_path, mode="r") as file:
#         reader = csv.reader(file)
#         next(reader)  # 헤더 스킵
#         for row in reader:
#             id = row[0]
#             embedding = np.array(eval(row[1]))  # 문자열을 리스트로 변환
#             embeddings[id] = embedding
#     return embeddings

# query_embeddings = load_embeddings("query_embeddings.csv")
# corpus_embeddings = load_embeddings("corpus_embeddings.csv")


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [ ]:
import csv
import asyncio
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import OpenAIEmbeddings

# 비동기 임베딩 벡터 생성 함수
async def aget_embedding(text, model="text-embedding-3-small"):
    embeddings = OpenAIEmbeddings(model=model)
    response = await embeddings.aembed_query(text)
    return response

# 쿼리와 코퍼스의 비동기 임베딩 생성 및 CSV 저장 함수
async def save_embeddings_async(queries, corpus):
    # 쿼리 임베딩 생성 및 저장
    with open("query_embeddings.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["query_id", "embedding"])
        query_tasks = [(qid, await aget_embedding(text)) for qid, text in queries]
        for qid, embedding in await asyncio.gather(*query_tasks):
            writer.writerow([qid, embedding])

    # 코퍼스 임베딩 생성 및 저장
    with open("corpus_embeddings.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["corpus_id", "embedding"])
        corpus_tasks = [(cid, await aget_embedding(f"{content['title']} {content['text']}"))
                        for cid, content in corpus]
        for cid, embedding in await asyncio.gather(*corpus_tasks):
            writer.writerow([cid, embedding])

# 임베딩 벡터 불러오기 함수
def load_embeddings(file_path):
    embeddings = {}
    with open(file_path, mode="r") as file:
        reader = csv.reader(file)
        next(reader)  # 헤더 스킵
        for row in reader:
            id = row[0]
            embedding = np.array(eval(row[1]))  # 문자열을 리스트로 변환
            embeddings[id] = embedding
    return embeddings

# 쿼리 및 코퍼스 데이터
queries = list(finder_task.queries.items())
corpus = list(finder_task.corpus.items())

# 비동기 이벤트 루프 실행
asyncio.run(save_embeddings_async(queries, corpus))

# 저장된 임베딩 파일 로드
query_embeddings = load_embeddings("query_embeddings.csv")
corpus_embeddings = load_embeddings("corpus_embeddings.csv")

print("임베딩 생성 및 저장이 완료되었습니다!")


In [15]:
# 유사도 계산 및 상위 10개 결과 추출
results = []
for qid, q_embed in query_embeddings.items():
    similarities = {}
    for cid, c_embed in corpus_embeddings.items():
        similarities[cid] = cosine_similarity([q_embed], [c_embed])[0][0]
    
    # 상위 10개 코퍼스 ID 추출
    top_10 = sorted(similarities, key=similarities.get, reverse=True)[:2]
    for cid in top_10:
        results.append((qid, cid))

# 결과를 CSV 파일로 저장
with open("query_corpus_matches.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["query_id", "corpus_id"])
    writer.writerows(results)

print("CSV 파일 'query_corpus_matches.csv'가 성공적으로 생성되었습니다!")

CSV 파일 'query_corpus_matches.csv'가 성공적으로 생성되었습니다!


In [26]:
import os
import csv
import asyncio
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from financerag.tasks import FinDER, FinQABench, FinanceBench, TATQA, FinQA, ConvFinQA, MultiHiertt
from langchain_openai import OpenAIEmbeddings

# 비동기 임베딩 생성 함수
async def aget_embedding(text, model="text-embedding-3-small"):
    embeddings = OpenAIEmbeddings(model=model)
    return await embeddings.aembed_query(text)

# 쿼리와 코퍼스 임베딩을 비동기적으로 생성하는 함수
async def embed_queries_and_corpus(queries, corpus):
    # 쿼리 임베딩 비동기 생성
    query_tasks = [(qid, aget_embedding(text)) for qid, text in queries]
    query_embeddings = [(qid, await embedding) for qid, embedding in query_tasks]

    # 코퍼스 임베딩 비동기 생성
    corpus_tasks = [(cid, aget_embedding(f"{content['title']} {content['text']}")) for cid, content in corpus]
    corpus_embeddings = [(cid, await embedding) for cid, embedding in corpus_tasks]

    return query_embeddings, corpus_embeddings

# 임베딩 결과를 CSV 파일에 저장하는 함수
def save_embeddings(task_name, query_embeddings, corpus_embeddings):
    os.makedirs(task_name, exist_ok=True)

    # 쿼리 임베딩 저장
    query_embeddings_path = os.path.join(task_name, "query_embeddings.csv")
    with open(query_embeddings_path, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["query_id", "embedding"])
        for qid, embedding in query_embeddings:
            writer.writerow([qid, embedding])

    # 코퍼스 임베딩 저장
    corpus_embeddings_path = os.path.join(task_name, "corpus_embeddings.csv")
    with open(corpus_embeddings_path, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["corpus_id", "embedding"])
        for cid, embedding in corpus_embeddings:
            writer.writerow([cid, embedding])

    return query_embeddings_path, corpus_embeddings_path

# 유사도 계산 및 상위 10개 매칭 결과 저장
def calculate_similarity_and_save(task_name, query_embeddings_path, corpus_embeddings_path):
    def load_embeddings(file_path):
        embeddings = {}
        with open(file_path, mode="r") as file:
            reader = csv.reader(file)
            next(reader)  # 헤더 스킵
            for row in reader:
                id = row[0]
                embedding = np.array(eval(row[1]))  # 문자열을 리스트로 변환
                embeddings[id] = embedding
        return embeddings
    
    query_embeddings = load_embeddings(query_embeddings_path)
    corpus_embeddings = load_embeddings(corpus_embeddings_path)
    
    results = []
    for qid, q_embed in query_embeddings.items():
        similarities = {}
        for cid, c_embed in corpus_embeddings.items():
            similarities[cid] = cosine_similarity([q_embed], [c_embed])[0][0]
        
        # 상위 10개 코퍼스 ID 추출
        top_10 = sorted(similarities, key=similarities.get, reverse=True)[:10]
        for cid in top_10:
            results.append((qid, cid))
    
    # 유사도 결과 저장
    output_path = os.path.join(task_name, "query_corpus_matches.csv")
    with open(output_path, mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["query_id", "corpus_id"])
        writer.writerows(results)
    
    return output_path

# 메인 함수
def main():
    tasks = {
        "FinDER": FinDER(),
        "FinQABench": FinQABench(),
        "FinanceBench": FinanceBench(),
        "TATQA": TATQA(),
        "FinQA": FinQA(),
        "ConvFinQA": ConvFinQA(),
        "MultiHiertt": MultiHiertt(),
    }
    
    similarity_results = []
    for task_name, task_instance in tasks.items():
        # 각 작업별 쿼리 및 코퍼스 데이터 준비
        queries = list(task_instance.queries.items())
        corpus = list(task_instance.corpus.items())

        # 각 작업별 비동기 임베딩 생성 및 저장
        query_embeddings, corpus_embeddings = asyncio.run(embed_queries_and_corpus(queries, corpus))
        query_embeddings_path, corpus_embeddings_path = save_embeddings(task_name, query_embeddings, corpus_embeddings)
        
        # 유사도 계산 및 결과 저장
        similarity_result_path = calculate_similarity_and_save(task_name, query_embeddings_path, corpus_embeddings_path)
        similarity_results.append(similarity_result_path)

    # 모든 작업의 유사도 결과 병합
    with open("all_query_corpus_matches.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["task_name", "query_id", "corpus_id"])
        
        for result_path in similarity_results:
            task_name = os.path.basename(os.path.dirname(result_path))
            with open(result_path, mode="r") as result_file:
                reader = csv.reader(result_file)
                next(reader)  # 헤더 스킵
                for row in reader:
                    writer.writerow([task_name] + row)

    print("모든 작업의 유사도 결과가 'all_query_corpus_matches.csv'에 병합되었습니다!")

# 실행
main()


INFO:financerag.common.loader:Loading Corpus...
INFO:financerag.common.loader:Loaded 13867 Documents.
INFO:financerag.common.loader:Corpus Example: {'id': 'ADBE20230004', 'title': 'ADBE OVERVIEW', 'text': 'Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsi

RuntimeError: asyncio.run() cannot be called from a running event loop

In [27]:
import csv
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
finder_task = FinDER()

queries = list(finder_task.queries.items())
corpus = list(finder_task.corpus.items())   

# OpenAI 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings()

# corpus 텍스트에 대해 임베딩을 생성하고 FAISS 데이터베이스로 만듭니다.
corpus_embeddings = {c[0]: embeddings.embed_query(c[1]["title"] + " " + c[1]["text"]) for c in corpus}
corpus_db = FAISS.from_embeddings(list(corpus_embeddings.values()), list(corpus_embeddings.keys()))

# FAISS 데이터베이스를 파일로 저장합니다.
corpus_db.save_local("corpus_faiss_index")

# 쿼리에 대한 임베딩을 생성하고, 각 쿼리에 대해 가장 유사한 코퍼스 ID를 CSV로 저장합니다.
with open("query_corpus_matches.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["query_id", "corpus_id"])  # CSV 헤더 작성

    for query_id, query_text in queries:
        query_embed = embeddings.embed_query(query_text)
        similar_docs = corpus_db.similarity_search_by_vector(query_embed, k=2)  # 유사한 항목 2개 검색

        for doc in similar_docs:
            writer.writerow([query_id, doc.metadata])  # 결과를 CSV에 저장



INFO:financerag.common.loader:Loading Corpus...
INFO:financerag.common.loader:Loaded 13867 Documents.
INFO:financerag.common.loader:Corpus Example: {'id': 'ADBE20230004', 'title': 'ADBE OVERVIEW', 'text': 'Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsi

KeyboardInterrupt: 

In [28]:
import csv
from tqdm import tqdm
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩을 생성합니다.
embeddings = OpenAIEmbeddings()

# corpus 텍스트에 대해 임베딩을 생성하고 FAISS 데이터베이스로 만듭니다.
corpus_embeddings = {}
for c in tqdm(corpus, desc="Embedding corpus texts"):
    corpus_embeddings[c[0]] = embeddings.embed_query(c[1]["title"] + " " + c[1]["text"])

corpus_db = FAISS.from_embeddings(list(corpus_embeddings.values()), list(corpus_embeddings.keys()))

# FAISS 데이터베이스를 파일로 저장합니다.
corpus_db.save_local("corpus_faiss_index")

# 쿼리에 대한 임베딩을 생성하고, 각 쿼리에 대해 가장 유사한 코퍼스 ID를 CSV로 저장합니다.
with open("query_corpus_matches.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["query_id", "corpus_id"])  # CSV 헤더 작성

    for query_id, query_text in tqdm(queries, desc="Processing queries"):
        query_embed = embeddings.embed_query(query_text)
        similar_docs = corpus_db.similarity_search_by_vector(query_embed, k=2)  # 유사한 항목 2개 검색

        for doc in similar_docs:
            writer.writerow([query_id, doc.metadata])  # 결과를 CSV에 저장


Embedding corpus texts:   1%|          | 88/13863 [00:31<1:22:29,  2.78it/s]
c:\Users\Jeawon\Desktop\2024데이터경진\data-contest-2024F\.venv\Lib\site-packages\pygments\regexopt.py:26: RuntimeWarning: coroutine 'embed_queries_and_corpus' was never awaited
  def regex_opt_inner(strings, open_paren):


KeyboardInterrupt: 

In [43]:
import nltk
from nltk.tokenize import sent_tokenize

# 테스트 코드
text = "This is a test. This is another test."
sentences = sent_tokenize(text, language='english')
print(sentences)


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\Jeawon/nltk_data'
    - 'c:\\Users\\Jeawon\\Desktop\\2024데이터경진\\data-contest-2024F\\.venv\\nltk_data'
    - 'c:\\Users\\Jeawon\\Desktop\\2024데이터경진\\data-contest-2024F\\.venv\\share\\nltk_data'
    - 'c:\\Users\\Jeawon\\Desktop\\2024데이터경진\\data-contest-2024F\\.venv\\lib\\nltk_data'
    - 'C:\\Users\\Jeawon\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - 'C:\\Users\\Jeawon\\AppData\\Roaming\\nltk_data'
    - 'C:\\Users\\Jeawon\\AppData\\Roaming\\nltk_data'
    - 'C:\\Users\\Jeawon\\AppData\\Roaming\\nltk_data'
    - 'C:/Users/Jeawon/Desktop/2

In [42]:
import nltk
nltk.data.path.append('C:/Users/Jeawon/Desktop/2024데이터경진/data-contest-2024F/.venv/nltk_data')
nltk.data.path.append('C:/Users/Jeawon/Desktop/2024데이터경진/data-contest-2024F/.venv/share/nltk_data')
nltk.data.path.append('C:/Users/Jeawon/Desktop/2024데이터경진/data-contest-2024F/.venv/lib/nltk_data')

# `punkt` 데이터를 가상 환경 경로에 설치
nltk.download('punkt', download_dir='C:/nltk_data')


[nltk_data] Downloading package punkt to C:/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True